In [ ]:
# PARAMETERS — edit this cell only.
from __future__ import annotations

import os
import sys
from pathlib import Path


def locate_bundle() -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "data" / "example_tiles.csv").is_file():
            return candidate
        nested = candidate / "tasks" / "260731_2_mllm_notebook_bundle"
        if (nested / "data" / "example_tiles.csv").is_file():
            return nested
    return Path.cwd().resolve()


BUNDLE_DIR = locate_bundle()
PROJECT_DIR = next(
    (candidate for candidate in [BUNDLE_DIR, *BUNDLE_DIR.parents]
     if (candidate / "outputs" / "runs").is_dir()),
    BUNDLE_DIR,
)
_open_source_default = PROJECT_DIR / "outputs/runs/open_source_all_tiles_current.csv"
_proprietary_default = PROJECT_DIR / "outputs/runs/proprietary_all_tiles_current.csv"
OPEN_SOURCE_FULL_CSV = Path(os.environ.get("OPEN_SOURCE_REFERENCE_CSV", _open_source_default if _open_source_default.is_file() else BUNDLE_DIR / "data/open_source_reference_sample.csv"))
PROPRIETARY_FULL_CSV = Path(os.environ.get("PROPRIETARY_REFERENCE_CSV", _proprietary_default if _proprietary_default.is_file() else BUNDLE_DIR / "data/proprietary_reference_sample.csv"))
SUMMARY_OUTPUT = BUNDLE_DIR / "data" / "reference_summary.csv"
RAW_SAMPLE_OUTPUT = BUNDLE_DIR / "data" / "reference_raw_sample.csv"
OPEN_SOURCE_SAMPLE_OUTPUT = BUNDLE_DIR / "data" / "open_source_reference_sample.csv"
PROPRIETARY_SAMPLE_OUTPUT = BUNDLE_DIR / "data" / "proprietary_reference_sample.csv"
SAMPLE_ROWS_PER_GROUP = 4
WRITE_REFERENCE_FILES = True


# 07 — Reference count data: summarize before sampling

The source exports are large, so the bundle does not copy them. This notebook derives a compact reference table from the full CSVs when they are available in the repository, then writes deterministic raw samples. The summary answers “roughly what should this model produce?” with per-model × per-task predicted and reference-count central tendencies, spread, mean absolute error, exact agreement, and parse-failure rate.

These are reference values from our completed runs, not ground truth for the model itself. Your own run should be compared against the same fields and row keys, not treated as an error merely because it differs from a reference model.


In [ ]:
import pandas as pd
from IPython.display import display

REFERENCE_COLUMNS = ["dataset", "task", "tileid", "ground_truth_label", "model_label", "nlg2choice_value", "final_value", "answer_parse_method", "answer_valid"]

def load_reference(path: Path, source_name: str) -> pd.DataFrame:
    if not path.is_file():
        raise FileNotFoundError(f"Reference CSV not found: {path}. Run this notebook from the repository or set its path in the parameter cell.")
    frame = pd.read_csv(path, usecols=REFERENCE_COLUMNS, low_memory=False)
    frame["reference_source"] = source_name
    frame["prediction_numeric"] = pd.to_numeric(frame["final_value"], errors="coerce")
    frame["ground_truth_numeric"] = pd.to_numeric(frame["ground_truth_label"], errors="coerce")
    return frame

open_source = load_reference(OPEN_SOURCE_FULL_CSV, "open_source")
proprietary = load_reference(PROPRIETARY_FULL_CSV, "proprietary")
reference = pd.concat([open_source, proprietary], ignore_index=True)
print({"open_source_input": str(OPEN_SOURCE_FULL_CSV), "proprietary_input": str(PROPRIETARY_FULL_CSV), "full_repository_exports_used": "outputs/runs" in str(OPEN_SOURCE_FULL_CSV) and "outputs/runs" in str(PROPRIETARY_FULL_CSV)})
reference["absolute_error"] = (reference["prediction_numeric"] - reference["ground_truth_numeric"]).abs()
reference["exact_match"] = reference["prediction_numeric"].eq(reference["ground_truth_numeric"])
reference["parse_failure"] = ~reference["answer_valid"].astype(bool)

summary = reference.groupby(["reference_source", "model_label", "dataset", "task"], as_index=False).agg(
    rows=("tileid", "size"),
    ground_truth_mean=("ground_truth_numeric", "mean"),
    ground_truth_median=("ground_truth_numeric", "median"),
    ground_truth_std=("ground_truth_numeric", "std"),
    predicted_mean=("prediction_numeric", "mean"),
    predicted_median=("prediction_numeric", "median"),
    predicted_std=("prediction_numeric", "std"),
    mae=("absolute_error", "mean"),
    exact_match_rate=("exact_match", "mean"),
    answer_valid_rate=("answer_valid", "mean"),
    parse_failure_rate=("parse_failure", "mean"),
).sort_values(["reference_source", "model_label", "dataset", "task"])
for column in ["ground_truth_mean", "ground_truth_median", "ground_truth_std", "predicted_mean", "predicted_median", "predicted_std", "mae", "exact_match_rate", "answer_valid_rate", "parse_failure_rate"]:
    summary[column] = summary[column].round(4)

raw_sample = reference.groupby(["reference_source", "model_label", "dataset", "task"], sort=True, group_keys=False).head(SAMPLE_ROWS_PER_GROUP).copy()
raw_sample = raw_sample[REFERENCE_COLUMNS + ["reference_source"]]
display(summary.head(12))
print({"full_rows": len(reference), "summary_rows": len(summary), "raw_sample_rows": len(raw_sample)})
if WRITE_REFERENCE_FILES:
    SUMMARY_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    summary.to_csv(SUMMARY_OUTPUT, index=False)
    raw_sample.to_csv(RAW_SAMPLE_OUTPUT, index=False)
    raw_sample[raw_sample["reference_source"] == "open_source"].to_csv(OPEN_SOURCE_SAMPLE_OUTPUT, index=False)
    raw_sample[raw_sample["reference_source"] == "proprietary"].to_csv(PROPRIETARY_SAMPLE_OUTPUT, index=False)


## Visualize the reference ranges

Use the summary to compare model families without loading the full row-level exports. The horizontal bars show mean predicted count next to mean reference count for the first task; the second panel shows spread and agreement rate across all model/task groups.


In [ ]:
from mu.visv2.figure import Figure

task = "birds/bird"
task_summary = summary[(summary["dataset"] + "/" + summary["task"] == task)].copy()
fig = Figure(n_rows=1, n_cols=2, figsize=(13, 4.8), fontfamily="sans-serif", fontsize=11, panel_padding=(0.3, 0.2))
left, right = fig.panels
labels = [f"{source}: {model}" for source, model in zip(task_summary["reference_source"], task_summary["model_label"])]
positions = list(range(len(labels)))
left.p.errorbar(task_summary["ground_truth_mean"], positions, xerr=task_summary["ground_truth_std"].fillna(0), fmt="o", color="#f59e0b", label="reference count")
left.p.errorbar(task_summary["predicted_mean"], positions, xerr=task_summary["predicted_std"].fillna(0), fmt="o", color="#2563eb", label="model count")
left.p.set_yticks(positions, labels, fontsize=7)
left.set_xaxislabel("mean count ± standard deviation")
left.set_title(f"Reference ranges: {task}")
left.p.legend(frameon=False, fontsize=8)
right.p.scatter(summary["mae"], summary["exact_match_rate"], s=16, alpha=0.4, color="#7c3aed")
right.set_xaxislabel("mean absolute error")
right.set_yaxislabel("exact-match rate")
right.set_title("Agreement across all model/task groups")
fig.show()


The four shipped reference files are the only count-data artifacts in this folder: one combined summary, one combined raw sample, and source-separated copies of that same small sample. Re-run this derivation when the full source exports change.
